# N093 · 贪心选择与交换论证

**目标：** 用局部替换证明而不是用样例支持贪心。

**先修：** N035, N013, N016。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 最早结束；交换论证；最优解保持；排序策略反例。

## 从问题到算法

区间调度的局部选择不是“最短区间”，而是“最早结束”。更早结束为后续保留不少于其他选择的空间。注意接口边界不同：不重叠区间允许端点接触；数对链要求前一终点严格小于后一开头。

## 最小实现

**本章接口：** `erase_overlap_intervals(intervals)`、`find_longest_chain(pairs)`

In [1]:
def erase_overlap_intervals(intervals):
    end=float('-inf'); kept=0
    for a,b in sorted(intervals,key=lambda x:x[1]):
        if a>=end: kept+=1; end=b
    return len(intervals)-kept

def find_longest_chain(pairs):
    end=float('-inf'); length=0
    for a,b in sorted(pairs,key=lambda x:x[1]):
        if a>end: length+=1; end=b
    return length

## 正确性、前提与复杂度

令最优方案首区间为O，最早结束区间为G。用G替换O不会与后续区间冲突，因为G结束不晚于O。因此存在以G开始的最优方案，对剩余区间归纳即可。严格与非严格比较必须和题意一致。

**代价：** O(n log n)时间、O(n)排序副本空间。输入每段满足start<end；不修改输入。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

intervals=[[1,2],[2,3],[3,4],[1,3]]; end=float('-inf'); trace=[]
for a,b in sorted(intervals,key=lambda x:x[1]):
    take=a>=end
    if take: end=b
    trace.append((a,b,take,end))
show_table(['起点','终点','保留','已选终点'],trace)

起点,终点,保留,已选终点
1,2,True,2
2,3,True,3
1,3,False,3
3,4,True,4


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert erase_overlap_intervals([[1,2],[2,3],[3,4],[1,3]])==1
assert find_longest_chain([[1,2],[2,3],[3,4]])==2
from itertools import combinations
from random import Random
rng=Random(93)
for _ in range(120):
    a=[sorted(rng.sample(range(8),2)) for _ in range(rng.randrange(8))]
    def opt(strict):
        best=0
        for k in range(len(a)+1):
            for group in combinations(a,k):
                seq=sorted(group)
                if all((seq[i][1]<seq[i+1][0] if strict else seq[i][1]<=seq[i+1][0]) for i in range(k-1)): best=max(best,k)
        return best
    assert erase_overlap_intervals(a)==len(a)-opt(False)
    assert find_longest_chain(a)==opt(True)
print('N093: 所有本章断言通过')

N093: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 反驳按最短区间优先。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 明确相邻端点在不同题目的兼容条件。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 435. Non-overlapping Intervals（canonical）：[官方入口](https://leetcode.com/problems/non-overlapping-intervals/)
- 646. Maximum Length of Pair Chain（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。